<a href="https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/solutions/Session14_hugging_face_transformers_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۴: مدل‌های ازپیش‌آموزش‌دیده با Hugging Face: تحلیل احساس متن فارسی با ParsBERT

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session14_hugging_face_transformers.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

> **نکتهٔ اجرا:** بعضی پاسخ‌ها مدل‌های بزرگ‌تری آموزش می‌دهند؛ در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
%pip install -q transformers datasets

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
print('device:', device)

PARSBERT = 'HooshvareLab/bert-fa-base-uncased'
LABELS = ['negative', 'positive', 'neutral']
id2label = dict(enumerate(LABELS))
label2id = {v: k for k, v in id2label.items()}
raw = load_dataset('hezarai/sentiment-dksf')
split = raw['train'].train_test_split(test_size=0.1, seed=42)
datasets_ = {'train': split['train'], 'validation': split['test'], 'test': raw['test']}


def make_loaders(model_name, max_len=128, batch_size=32):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tok = {name: ds.map(lambda b: tokenizer(b['text'], truncation=True, max_length=max_len), batched=True,
                        remove_columns=['text']).with_format('torch') for name, ds in datasets_.items()}
    collator = DataCollatorWithPadding(tokenizer)
    loaders = {name: DataLoader(ds, batch_size=batch_size if name == 'train' else 128, shuffle=name == 'train',
                                collate_fn=collator) for name, ds in tok.items()}
    return tokenizer, loaders


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, labels = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**{k: v for k, v in batch.items() if k != 'labels'}).logits
        preds.append(logits.argmax(-1).cpu()); labels.append(batch['labels'].cpu())
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


def fine_tune(model_name, loaders, lr=3e-5, epochs=2, freeze_body=False, label=''):
    torch.manual_seed(42)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3, id2label=id2label, label2id=label2id).to(device)
    if freeze_body:                                          # train only the classification head
        for name, p in model.named_parameters():
            p.requires_grad = name.startswith('classifier')
    no_decay = ('bias', 'LayerNorm.weight')
    trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    groups = [{'params': [p for n, p in trainable if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
              {'params': [p for n, p in trainable if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}]
    optimizer = torch.optim.AdamW(groups, lr=lr)
    total = epochs * len(loaders['train'])
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * total), total)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    start = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        for batch in loaders['train']:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                loss = model(**batch).loss
            optimizer.zero_grad()
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
        p, y = predict(model, loaders['validation'])
        print(f'{label} epoch {epoch}: val accuracy {accuracy_score(y, p):.4f} | val macro-F1 {f1_score(y, p, average="macro"):.4f}')
    print(f'{label}: {time.time() - start:.0f}s')
    return model


parsbert_tokenizer, parsbert_loaders = make_loaders(PARSBERT)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

نرخ یادگیری را $10^{-5}$ و $10^{-4}$ بگیرید و Macro-F1 اعتبارسنجی را مقایسه کنید. با نرخ بزرگ چه اتفاقی می‌افتد؟

</div>

In [ ]:
for lr in (1e-5, 3e-5, 1e-4):
    fine_tune(PARSBERT, parsbert_loaders, lr=lr, label=f'lr={lr:.0e}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

- با $10^{-5}$ آموزش پایدار است، ولی در ۲ Epoch کمی کمتر از $3\times10^{-5}$ پیش می‌رود.
- با $10^{-4}$ قدم‌ها برای مدلی با ۱۶۰ میلیون پارامتر پیش‌آموزش‌دیده بزرگ‌اند. معمولاً نتیجه بدتر است و گاهی آموزش کاملاً **فرو می‌ریزد**: مدل همهٔ نمونه‌ها را در کلاس اکثریت می‌گذارد و Macro-F1 به حدود ۰٫۲۵ سقوط می‌کند. این همان **Catastrophic Forgetting** است: به‌روزرسانی‌های بزرگ دانش زبانی لایه‌های پیش‌آموزش‌دیده را پاک می‌کنند.

به همین دلیل محدودهٔ $2\times10^{-5}$ تا $5\times10^{-5}$ پیشنهاد استاندارد Fine-tuning مدل‌های BERT است. نتیجهٔ دقیق به Seed هم بستگی دارد؛ در گزارش‌های جدی میانگین چند اجرا را بدهید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

مدل چندزبانه `xlm-roberta-base` را به‌جای ParsBERT Fine-tune کنید و نتیجه را مقایسه کنید.

</div>

In [ ]:
xlmr_tokenizer, xlmr_loaders = make_loaders('FacebookAI/xlm-roberta-base')
sample = 'کیفیت گوشی عالی بود ولی ارسالش خیلی دیر انجام شد'
print('ParsBERT tokens:', parsbert_tokenizer.tokenize(sample))
print('XLM-R tokens   :', xlmr_tokenizer.tokenize(sample))
xlmr = fine_tune('FacebookAI/xlm-roberta-base', xlmr_loaders, lr=2e-5, epochs=3, label='XLM-R')
p, y = predict(xlmr, xlmr_loaders['test'])
print(f'XLM-R test accuracy {accuracy_score(y, p):.4f} | macro-F1 {f1_score(y, p, average="macro"):.4f} | '
      f'parameters {sum(q.numel() for q in xlmr.parameters()) / 1e6:.0f}M')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

XLM-RoBERTa روی متن ۱۰۰ زبان (از جمله فارسی) پیش‌آموزش دیده است و حدود ۲۸۰ میلیون پارامتر دارد. بیشتر این پارامترها در جدول Embedding واژگان ۲۵۰ هزارتایی آن است. نتیجه معمولاً هم‌سطح ParsBERT است: ParsBERT مزیت تخصص در فارسی و Tokenizer مخصوص فارسی را دارد. XLM-R مزیت دادهٔ پیش‌آموزش بسیار بزرگ‌تر را دارد و یک مدل برای همهٔ زبان‌هاست. XLM-R گاهی به نرخ یادگیری کوچک‌تر و Epoch بیشتر نیاز دارد تا پایدار شود؛ برای همین $2\times10^{-5}$ و ۳ Epoch گرفتیم. Tokenهای XLM-R با `▁` (شروع کلمه در SentencePiece) علامت‌گذاری می‌شوند، نه `##`.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

فقط سر طبقه‌بند را آموزش دهید و بدنه BERT را منجمد کنید (مثل Feature Extraction در جلسه ۹). چقدر افت دقت دارد؟

</div>

In [ ]:
head_only = fine_tune(PARSBERT, parsbert_loaders, lr=1e-3, epochs=3, freeze_body=True, label='frozen body')
trainable = sum(p.numel() for p in head_only.parameters() if p.requires_grad)
print(f'trainable parameters: {trainable:,}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با بدنهٔ منجمد فقط لایهٔ طبقه‌بند ($768 \times 3 + 3$ پارامتر) آموزش می‌بیند. چون این لایه کوچک و از صفر است، نرخ یادگیری بزرگ‌تر ($10^{-3}$) لازم است. افت دقت معمولاً **قابل توجه** است، بسیار بیشتر از Feature Extraction در جلسهٔ ۹. دلیل: بردار `[CLS]` در BERT برای پیش‌بینی کلمهٔ ماسک‌شده و ترتیب جمله‌ها آموزش دیده است، نه برای خلاصه کردن احساس جمله. برخلاف ویژگی‌های ImageNet که برای طبقه‌بندی آموزش دیده‌اند، ویژگی‌های BERT برای طبقه‌بندی آماده نیستند و به Fine-tuning نیاز دارند. استخراج ویژگی از BERT با میانگین بردار همهٔ Tokenها (Mean Pooling) معمولاً نتیجهٔ بهتری از `[CLS]` تنها می‌دهد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

با `MAX_LEN = 64` آموزش دهید. زمان آموزش و دقت چه تغییری می‌کند؟ با هیستوگرام طول نظرات (بر حسب Token) تصمیم بگیرید چه طولی مناسب است.

</div>

In [ ]:
lengths = [len(ids) for ids in parsbert_tokenizer(datasets_['train']['text'])['input_ids']]
print(f'tokens per review: median {np.median(lengths):.0f} | 90th percentile {np.percentile(lengths, 90):.0f} | '
      f'95th {np.percentile(lengths, 95):.0f} | longer than 64: {np.mean(np.array(lengths) > 64):.1%}')
plt.hist(np.clip(lengths, 0, 200), bins=50)
plt.axvline(64, color='C1', ls='--', label='64'); plt.axvline(128, color='C2', ls='--', label='128')
plt.xlabel('tokens per review (clipped at 200)'); plt.ylabel('reviews'); plt.legend(); plt.show()

_, short_loaders = make_loaders(PARSBERT, max_len=64)
fine_tune(PARSBERT, short_loaders, label='MAX_LEN=64')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

نظرات کاربران دیجی‌کالا و اسنپ‌فود بیشتر کوتاه‌اند و فقط بخش کوچکی از آن‌ها بیش از ۶۴ Token دارند (عدد دقیق در خروجی بالا). پس با `MAX_LEN = 64` فقط تعداد کمی نظر کوتاه می‌شوند و دقت تقریباً تغییر نمی‌کند. زمان آموزش هم به لطف **Dynamic Padding** فقط کمی کم می‌شود: Batchها از قبل فقط تا طول بلندترین نمونهٔ همان Batch پر می‌شدند. **قاعده:** طولی را انتخاب کنید که حدود ۹۵٪ نمونه‌ها را کامل پوشش دهد؛ هزینهٔ Attention با مربع طول رشد می‌کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

مدل آموزش‌دیده را با `push_to_hub` در حساب Hugging Face خودتان منتشر کنید و یک Model Card برای آن بنویسید.

</div>

In [ ]:
from huggingface_hub import notebook_login

notebook_login()                       # paste a token with "write" permission from huggingface.co/settings/tokens

final_model = fine_tune(PARSBERT, parsbert_loaders, label='final')
p, y = predict(final_model, parsbert_loaders['test'])
test_acc, test_f1 = accuracy_score(y, p), f1_score(y, p, average='macro')

REPO_ID = 'your-username/parsbert-sentiment-dksf'         # change to your Hugging Face username
final_model.push_to_hub(REPO_ID)
parsbert_tokenizer.push_to_hub(REPO_ID)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**Model Card** فایل `README.md` مخزن مدل است. Hub آن را در صفحهٔ مدل نمایش می‌دهد. بخش بالای آن (YAML) به Hub می‌گوید زبان، مجوز، دیتاست و معیارهای مدل چیست:

</div>

In [ ]:
from huggingface_hub import ModelCard

card = ModelCard(f"""---
language: fa
license: apache-2.0
base_model: {PARSBERT}
datasets:
- hezarai/sentiment-dksf
metrics:
- accuracy
- f1
pipeline_tag: text-classification
---

# ParsBERT fine-tuned for Persian sentiment analysis

Classifies Persian product and food-delivery reviews into **negative**, **positive** or **neutral**.

## Training
- Base model: `{PARSBERT}`
- Data: `hezarai/sentiment-dksf` (Digikala and Snappfood reviews), 10% of the training split held out for validation
- 2 epochs, AdamW, learning rate 3e-5, linear warmup and decay, max length 128 tokens

## Results (test split)
| accuracy | macro-F1 |
|---|---|
| {test_acc:.3f} | {test_f1:.3f} |

## Limitations
- Trained on short e-commerce and food reviews; other domains (news, tweets, formal text) may perform worse.
- The neutral class is the hardest and is often confused with the other two.
""")
card.push_to_hub(REPO_ID)
print(f'https://huggingface.co/{REPO_ID}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

بعد از انتشار، هر کسی می‌تواند مدل را با یک خط استفاده کند: `pipeline('text-classification', model=REPO_ID)`. Model Card خوب باید بگوید مدل برای چه کاری ساخته شده، روی چه داده‌ای آموزش دیده، چه نتیجه‌ای گرفته و **کجا نباید** به آن اعتماد کرد.

</div>